In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_T_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "53234477 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin1,2,0,1
42667,96.7,28.3,T0DayMin1,2,0,1


In [3]:
df_deg_bins%>%count(Day_Min)

Day_Min,n
<int>,<int>
1,1766072
2,1766439
3,1766947
4,1768303
5,1768954
6,1768375
7,1768557
8,1769323
9,1769276


In [4]:
df_deg_bins%>%count(N_hours)

N_hours,n
<int>,<int>
1,18442708
2,17775433
3,8254867
4,4304199
5,2011538
6,964596
7,531993
8,327213
9,207260


In [5]:
df_deg_bins%>%count(Temp)

Temp,n
<int>,<int>
-27,8
-26,15
-25,45
-24,81
-23,105
-22,156
-21,241
-20,323
-19,477


# Computation $$T_{i}^{30}= \sum_{b} \frac{h_{ib}^{30} \times T_b}{24 \times 30}$$

## 30-day exposure

In [6]:
result_30 <- df_deg_bins[, .(T_i30 = sum(N_hours * Temp / 24 / 30, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_30), ncol(result_30))
head(result_30,2)

[1] "173657 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,T_i30
<int>,<dbl>,<dbl>,<dbl>
42474,91.9,27.6,3.495833
42667,96.7,28.3,9.127778


## 20-day exposure 

In [7]:
df_deg_bins_20 <- df_deg_bins[Day_Min <=20]
sprintf("%i x %i dataframe", nrow(df_deg_bins_20), ncol(df_deg_bins_20))
head(df_deg_bins,2)

[1] "35416118 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin1,2,0,1
42667,96.7,28.3,T0DayMin1,2,0,1


In [8]:
result_20 <- df_deg_bins_20[, .(T_i20 = sum(N_hours * Temp / 24 / 20, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_20), ncol(result_20))
head(result_20,2)

[1] "173657 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,T_i20
<int>,<dbl>,<dbl>,<dbl>
42474,91.9,27.6,4.510417
42667,96.7,28.3,8.314583


## 10-day exposure

In [9]:
df_deg_bins_10 <- df_deg_bins[Day_Min <=10]
sprintf("%i x %i dataframe", nrow(df_deg_bins_10), ncol(df_deg_bins_10))
head(df_deg_bins_10,2)

[1] "17682220 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin1,2,0,1
42667,96.7,28.3,T0DayMin1,2,0,1


In [10]:
result_10 <- df_deg_bins_10[, .(T_i10 = sum(N_hours * Temp / 24 / 10, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_10), ncol(result_10))
head(result_10,2)

[1] "173657 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,T_i10
<int>,<dbl>,<dbl>,<dbl>
42474,91.9,27.6,5.112500
42667,96.7,28.3,7.016667


## MERGE

In [11]:
result <- data.frame(result_30) %>%
                left_join(result_20,by=c("Measured_date_CDC","cell_x","cell_y"))%>%
                left_join(result_10,by=c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "173657 x 6 dataframe"

,Measured_date_CDC,cell_x,cell_y,T_i30,T_i20,T_i10
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,42474,91.9,27.6,3.495833,4.510417,5.112500
2,42667,96.7,28.3,9.127778,8.314583,7.016667


In [12]:
data.frame(result) %>% 
        select(T_i30,T_i20,T_i10)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic    N    Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
----------------------------------------------------------------------
T_i30     173,657 22.96   6.49   -14.62  18.72   24.27   27.61   37.43
T_i20     173,657 23.12   6.46   -15.53  19.05   24.47   27.66   37.48
T_i10     173,657 23.31   6.41   -16.78  19.41   24.55   27.74   38.16
----------------------------------------------------------------------


# Categorize temperature into bins

### Air temperature thresholds

In [13]:
thresholds_T <- c(-17,17,21,23,25,27,29,31,33,40)

In [14]:
T_bins_labels <- c("<=17°C","(17,21]","(21,23]","(23,25]","(25,27]","(27,29]","(29,31]","(31,33]",">33°C")

In [15]:
result_bins <- result %>%
                    mutate(T_i30_bin = cut(T_i30, breaks = thresholds_T, 
                                           labels = T_bins_labels ,
                                           include.lowest = TRUE),
                           T_i20_bin = cut(T_i20, breaks = thresholds_T, 
                                           labels = T_bins_labels ,
                                           include.lowest = TRUE),
                           T_i10_bin = cut(T_i10, breaks = thresholds_T, 
                                           labels = T_bins_labels ,
                                           include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(result_bins), ncol(result_bins))
head(result_bins,2)

[1] "173657 x 9 dataframe"

,Measured_date_CDC,cell_x,cell_y,T_i30,T_i20,T_i10,T_i30_bin,T_i20_bin,T_i10_bin
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>
1,42474,91.9,27.6,3.495833,4.510417,5.112500,<=17°C,<=17°C,<=17°C
2,42667,96.7,28.3,9.127778,8.314583,7.016667,<=17°C,<=17°C,<=17°C


## SAVE

In [18]:
fwrite(result_bins,"./output/df_T_cells_CDC_30_20_10.csv")